In [1]:
import pandas as pd 

In [2]:
human_tfs_df = pd.read_csv("./human/01_cyt_nuc_TFs/human_proteome_TFs.csv", index_col=0)
human_cytoskeletal_df = pd.read_csv("./human/01_cyt_nuc_TFs/uniprot-compressed_cytoskeleton_HS.csv")
human_nucleus_df = pd.read_csv("./human/01_cyt_nuc_TFs/uniprot-compressed_nucleus_HS.csv")

**Combine Cyt, Nuc and TF files**

In [3]:
df_to_combine_list = [human_tfs_df, human_cytoskeletal_df, human_nucleus_df]

In [4]:
for df in df_to_combine_list:
    print(df.shape)
    
for df in df_to_combine_list:
    print(df.columns)

(6363, 12)
(1333, 16)
(5518, 16)
Index(['Entry', 'Reviewed', 'Entry Name', 'Protein names', 'Gene Names',
       'Organism', 'Length', 'Gene Ontology (biological process)',
       'Gene Ontology (GO)', 'Gene Ontology IDs',
       'Gene Ontology (cellular component)',
       'Gene Ontology (molecular function)'],
      dtype='object')
Index(['Entry', 'Reviewed', 'Entry Name', 'Protein names', 'Gene Names',
       'Organism', 'Length', 'Gene Ontology (cellular component)',
       'Subcellular location [CC]', 'Mass', 'Function [CC]',
       'Tissue specificity', 'Gene Ontology (biological process)',
       'Gene Ontology (molecular function)', 'Signal peptide',
       'Transit peptide'],
      dtype='object')
Index(['Entry', 'Reviewed', 'Entry Name', 'Protein names', 'Gene Names',
       'Organism', 'Length', 'Gene Ontology (cellular component)',
       'Subcellular location [CC]', 'Mass', 'Function [CC]',
       'Tissue specificity', 'Gene Ontology (biological process)',
       'Gene Ont

In [5]:
col_list = ['Entry', 'Reviewed', 'Entry Name', 'Protein names', 'Gene Names',
       'Organism', 'Length', 'Gene Ontology (cellular component)',
       'Subcellular location [CC]', 'Mass', 'Function [CC]',
       'Tissue specificity', 'Gene Ontology (biological process)',
       'Gene Ontology (molecular function)', 'Signal peptide',
       'Transit peptide']

cyt_nuc_df = human_cytoskeletal_df.merge(human_nucleus_df, on=col_list, how='outer')
cyt_nuc_df.shape

(6461, 16)

In [6]:
col_list = ['Entry', 'Reviewed', 'Entry Name', 'Protein names', 'Gene Names',
       'Organism', 'Length', 'Gene Ontology (cellular component)',
       'Gene Ontology (biological process)',
       'Gene Ontology (molecular function)']

cyt_nuc_tfs_df = cyt_nuc_df.merge(human_tfs_df, on=col_list, how='outer')
cyt_nuc_tfs_df.shape

(11454, 18)

**Combine secreted lists**

In [11]:
human_secreted_df = pd.read_csv("./human/02_secreted/uniprot-compressed_Secreted_HS_08032022.csv")
human_secreted_hpa_df = pd.read_csv("./human/02_secreted/HPA_HS_secreted.csv")

#ER
human_secreted_subcell_loc_hpa_df = pd.read_csv("./human/02_secreted/subcell_location_Endoplasmic_HumanProteinAtlas.csv")


In [12]:
human_secreted_to_combine_list = [human_secreted_df, human_secreted_hpa_df, human_secreted_subcell_loc_hpa_df]

In [13]:
human_secreted_hpa_df.rename(columns = {'Uniprot':'Entry'}, inplace = True)
human_secreted_subcell_loc_hpa_df.rename(columns = {'Uniprot':'Entry'}, inplace = True)

In [14]:
for df in human_secreted_to_combine_list:
    print(df.shape)
    
for df in human_secreted_to_combine_list:
    print(df.columns)

(2096, 15)
(1903, 315)
(523, 315)
Index(['Entry', 'Reviewed', 'Entry Name', 'Protein names', 'Gene Names',
       'Organism', 'Length', 'Gene Ontology (cellular component)',
       'Subcellular location [CC]', 'Mass', 'Function [CC]',
       'Tissue specificity', 'Gene Ontology (biological process)',
       'Signal peptide', 'Transit peptide'],
      dtype='object')
Index(['Gene', 'Gene synonym', 'Ensembl', 'Gene description', 'Entry',
       'Chromosome', 'Position', 'Protein class', 'Biological process',
       'Molecular function',
       ...
       'Single Cell Type RNA - Smooth muscle cells [nTPM]',
       'Single Cell Type RNA - Spermatocytes [nTPM]',
       'Single Cell Type RNA - Spermatogonia [nTPM]',
       'Single Cell Type RNA - Squamous epithelial cells [nTPM]',
       'Single Cell Type RNA - Suprabasal keratinocytes [nTPM]',
       'Single Cell Type RNA - Syncytiotrophoblasts [nTPM]',
       'Single Cell Type RNA - T-cells [nTPM]',
       'Single Cell Type RNA - Theca cel

In [16]:
col_list = human_secreted_hpa_df.columns.tolist()
hpa_merged = human_secreted_subcell_loc_hpa_df.merge(human_secreted_hpa_df, on=col_list, how='outer')
hpa_merged.shape

(2366, 315)

In [17]:
secreted_hpa_combined_df = human_secreted_df.merge(hpa_merged, on="Entry", how='outer')
secreted_hpa_combined_df.shape

(2669, 329)

In [18]:
secreted_hpa_combined_df.to_csv("./human/00_combined/human_TP_list_secreted.csv", index=False)

**Remove overlapping uniprot IDs with secreted df**

In [19]:
secreted_uniprot_list = secreted_hpa_combined_df["Entry"].tolist()
len(secreted_uniprot_list)

2669

In [20]:
unique_cyt_nuc_tfs_df = cyt_nuc_tfs_df[~cyt_nuc_tfs_df["Entry"].isin(secreted_uniprot_list)]

In [21]:
unique_cyt_nuc_tfs_df.shape

(11276, 18)

In [22]:
unique_cyt_nuc_tfs_df.to_csv("./human/00_combined/human_FP_list_cyt_nuc_TFs.csv", index=False)